# Multi-Hop Subliminal Learning — Analysis
Starting point for exploration. Uses `scripts.analysis` + the raw metrics.

In [ ]:
import os, sys
from pathlib import Path

# Find the repo root (dir containing scripts/) and work from there.
ROOT = Path.cwd()
while not (ROOT / 'scripts').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

import matplotlib.pyplot as plt
from scripts import analysis, paths, utils

FAMILY = 'qwen2.5-7b'
SEED = 0

## First-hop validation

In [ ]:
def trait_rate(root, hop, seed=SEED, family=FAMILY):
    p = paths.hop_dir(root, family, seed, hop) / 'metrics.json'
    return utils.read_json(p)['trait_rate'] if p.exists() else None

def trait_name(root, hop=0, seed=SEED, family=FAMILY):
    p = paths.hop_dir(root, family, seed, hop) / 'metadata.json'
    return utils.read_json(p).get('trait', 'trait') if p.exists() else 'trait'

bars = [('base baseline', trait_rate('data/validation-base', 0)),
        ('teacher (hop 0)', trait_rate('data/validation', 0)),
        ('student (hop 1)', trait_rate('data/validation', 1))]
bars = [(n, tr) for n, tr in bars if tr]
trait = trait_name('data/validation', 0)

fig, ax = plt.subplots(figsize=(5, 4))
labels = [n for n, _ in bars]
means = [tr['mean'] for _, tr in bars]
lo = [tr['mean'] - tr['ci_low'] for _, tr in bars]
hi = [tr['ci_high'] - tr['mean'] for _, tr in bars]
ax.bar(labels, means, yerr=[lo, hi], capsize=5,
       color=['#b0b0b0', '#4c72b0', '#dd8452'][:len(bars)])
ax.set_ylabel(f'{trait} rate'); ax.set_ylim(0, 1)
ax.set_title(f'First-hop validation ({FAMILY}, seed {SEED})')
plt.show()

## 3-hop validation
Run §5 (`run_chain --n-hops 3` on Vast) and download `data/validation-3hop`. The next cell builds the metrics locally; the rest plots them.

In [ ]:
import subprocess, sys
ROOT3 = 'data/validation-3hop'
# Build metrics.json + summary.parquet from the downloaded raw artifacts
# (run_analysis auto-detects the hop count). Skips if nothing is downloaded.
if (paths.base_reference_dir(ROOT3, FAMILY) / 'base_sequences.jsonl').exists():
    subprocess.run([sys.executable, '-m', 'scripts.run_analysis',
                    '--root', ROOT3, '--family', FAMILY, '--seed', str(SEED)], check=True)
else:
    print(f'No 3-hop artifacts under {ROOT3} — run §5 and pull them first.')

In [ ]:
have3 = (paths.seed_dir(ROOT3, FAMILY, SEED) / 'summary.parquet').exists()
df3 = analysis.load_all_summaries(ROOT3, FAMILY) if have3 else None
print('3-hop summary loaded.' if have3 else 'No 3-hop results — run the cell above.')
df3

In [ ]:
if have3:
    fig, axes = plt.subplots(2, 2, figsize=(11, 8)); a = axes.ravel()
    a[0].plot(df3['hop'], df3['trait_rate'], marker='o')
    a[0].set_title('trait rate'); a[0].set_xlabel('hop')
    a[1].plot(df3['hop'], df3['eas_last'], marker='o', color='C2')
    a[1].set_title('EAS (cos vs teacher direction)'); a[1].set_xlabel('hop')
    for col, mk in [('entangled_data', 'o'), ('entangled_unembedding', 's'),
                    ('entangled_logit', '^')]:
        a[2].plot(df3['hop'], df3[col], marker=mk, label=col.replace('entangled_', ''))
    a[2].set_title('entangled-token freq'); a[2].set_xlabel('hop'); a[2].legend()
    a[3].plot(df3['hop'], df3['divergence_freq_A'], marker='o', label='A: carrier freq')
    a[3].plot(df3['hop'], df3['divergence_rate_B'], marker='s', label='B: divergence rate')
    a[3].set_title('divergence tokens'); a[3].set_xlabel('hop'); a[3].legend()
    fig.tight_layout(); plt.show()

### Correlations & loss

In [ ]:
if have3:
    cols = ['eas_last', 'entangled_data', 'entangled_unembedding',
            'entangled_logit', 'divergence_freq_A', 'divergence_rate_B']
    display(analysis.correlation_table(df3, cols))  # vs trait_rate; only ~4 points here

In [ ]:
if have3:
    analysis.plot_loss_curves(ROOT3, FAMILY, SEED); plt.show()

## 5-hop trait trend (2 epochs)
Trait-only preview (§5.3). Reads per-hop `metrics.json` (trait rate) — no local analysis needed. `trait_score` ran on the instance.

In [ ]:
ROOT5 = 'data/validation-5hop-2epochs'
hops, rates, lo, hi = [], [], [], []
for hop in range(6):
    p = paths.hop_dir(ROOT5, FAMILY, SEED, hop) / 'metrics.json'
    tr = utils.read_json(p).get('trait_rate') if p.exists() else None
    if tr:
        hops.append(hop); rates.append(tr['mean'])
        lo.append(tr['mean'] - tr['ci_low']); hi.append(tr['ci_high'] - tr['mean'])
if hops:
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.errorbar(hops, rates, yerr=[lo, hi], marker='o', capsize=4)
    ax.set_xlabel('hop'); ax.set_ylabel('trait rate'); ax.set_ylim(0, 1)
    ax.set_title('5-hop trait trend (2 epochs)'); plt.show()
else:
    print(f'No 5-hop/2-epoch results under {ROOT5} - run §5.3 and pull them.')